# Slide: Defect Surface Site Analysis — Detecting Sites Inside the Void

Produces three trajectory files and matching figures for one slide:

| File | Panel | Description |
|---|---|---|
| `panel1_void_ring.traj` | Left | Slab + undercoordinated ring atoms (He) + void centroid (Ne) |
| `panel2_drop_steps.traj` | Middle | Noble-gas probe descending into void, one frame per z step |
| `panel3_void_sites.traj` | Right | One frame per unique probe position inside the void |

**Edit Cell 2 only, then Run All.**

In [1]:
# Cell 1: imports — do not edit
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from ase import Atoms
from ase.io import read, write, Trajectory
from ase.neighborlist import NeighborList
from ase.geometry import get_distances

import site_analysis as sa
import importlib
importlib.reload(sa)
print('site_analysis loaded')

ModuleNotFoundError: No module named 'yaml'

In [ ]:
# Cell 2: user inputs — edit here
xyz_path      = 'pt111_defect_middle.xyz'
tag_symbol    = 'Ne'   # noble-gas probe for the drop
dz            = 0.1    # Å per drop step
stable_steps  = 50     # stop when CN does not improve for this many steps (50 = 5 Å patience)
max_drop      = 25.0   # Å maximum total drop distance (must be > depth of void + 3 Å start offset)
min_clearance = 1.3    # Å minimum probe-to-slab distance before stopping
margin        = 0.25   # Å added to NN distance to set CN cutoff radius
uniq_tol      = 0.25   # Å tolerance for unique-position deduplication

## Panel 1 — Void ring: top-view of undercoordinated atoms

**Method:**
1. Identify top-layer atoms.
2. Build in-plane neighbor graph; atoms with degree ≤ median − 1 are undercoordinated.
3. Cluster undercoordinated atoms; their centroid = void position.

**Trajectory frames:**
- Frame 0: clean slab
- Frame 1: slab + **He** at each undercoordinated atom (0.5 Å above, for visibility)
- Frame 2: Frame 1 + **Ne** at void centroid

In [ ]:
# Panel 1 — void ring trajectory
slab  = read(xyz_path)
nslab = len(slab)

# top-surface indices and in-plane NN distance
top     = sa._top_surface_indices(slab, nslab)
use_pbc = sa._has_reasonable_cell(slab)
d_nn    = sa._estimate_nn_distance_xy(slab, top, use_pbc_xy=use_pbc)
cutoff  = 1.25 * d_nn

# build neighbor list and compute in-plane degrees for top atoms
nl = NeighborList([cutoff] * nslab, self_interaction=False, bothways=True, skin=0.0)
nl.update(slab)

top_set = set(top.tolist())
deg = np.zeros(len(top), dtype=int)
for ii, a in enumerate(top):
    neigh, _ = nl.get_neighbors(a)
    deg[ii]  = sum(1 for j in neigh if j in top_set)

expected   = int(np.median(deg))
under_mask = deg <= (expected - 1)
under_top  = top[under_mask]   # slab indices of the void-ring atoms

print(f'Top-layer atoms  : {len(top)}')
print(f'Expected degree  : {expected}')
print(f'Undercoordinated : {len(under_top)}')

# detect vacancy centroid
defect_sites, _ = sa.detect_vacancy_sites_from_coordination(slab, nslab)
if not defect_sites:
    raise RuntimeError('No vacancy detected. Check xyz_path or try a different slab.')
void_pos = defect_sites[0]['position']
print(f'\nVoid centroid: x={void_pos[0]:.3f}  y={void_pos[1]:.3f}  z={void_pos[2]:.3f}')

# build frames
frame0 = slab.copy()

frame1 = slab.copy()
for idx in under_top:
    pos      = slab.positions[idx].copy()
    pos[2]  += 0.5          # small z-offset so He sits above the Pt atom
    frame1  += Atoms('He', positions=[pos])

frame2  = frame1.copy()
frame2 += Atoms(tag_symbol, positions=[void_pos])

traj1 = Trajectory('panel1_void_ring.traj', 'w')
for f in [frame0, frame1, frame2]:
    traj1.write(f)
traj1.close()

print('\nWrote: panel1_void_ring.traj')
print('  Frame 0 – clean slab')
print('  Frame 1 – slab + He at undercoordinated ring atoms')
print('  Frame 2 – Frame 1 + Ne at void centroid')

In [ ]:
# Panel 1 — top-view matplotlib figure
top_pos   = slab.positions[top]
under_pos = slab.positions[under_top]

fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(top_pos[:, 0], top_pos[:, 1],
           c='steelblue', s=220, edgecolors='k', linewidths=0.6,
           zorder=2, label='surface atoms')

ax.scatter(under_pos[:, 0], under_pos[:, 1],
           c='tomato', s=260, edgecolors='k', linewidths=1.2,
           zorder=3, label=f'undercoordinated (deg ≤ {expected-1})')

ax.scatter(void_pos[0], void_pos[1],
           c='gold', s=350, marker='*', edgecolors='k', linewidths=0.9,
           zorder=4, label='void centroid')

ax.set_xlabel('x (Å)', fontsize=12)
ax.set_ylabel('y (Å)', fontsize=12)
ax.set_title('Panel 1: Top view — void ring detection', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('panel1_void_ring.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel1_void_ring.png')

## Panel 2 — Drop trajectory: probe descending into the void

**Method:**
1. Place noble-gas probe 3 Å above the void centroid.
2. Lower probe by `dz` each step; count coordination number (CN) to slab atoms.
3. Stop when CN is stable for `stable_steps` steps, or probe is too close to slab.

**Trajectory:** one frame per z step (probe moving into void).  
**Figure:** CN vs z, with max-CN step highlighted.

In [ ]:
# Panel 2 — drop trajectory
NOBLE_GASES = {'He', 'Ne', 'Ar', 'Kr', 'Xe', 'Rn'}

def _estimate_first_nn(atoms):
    idx = [i for i, a in enumerate(atoms) if a.symbol not in NOBLE_GASES]
    if len(idx) < 2:
        return 3.0
    pos = atoms.positions[idx]
    _, dmat = get_distances(pos, pos, cell=atoms.cell, pbc=atoms.pbc)
    dmat = np.array(dmat)
    np.fill_diagonal(dmat, np.inf)
    nn = np.min(dmat, axis=1)
    nn = nn[np.isfinite(nn)]
    return float(np.median(nn)) if len(nn) else 3.0

def _tag_cn(atoms, i_tag, cutoff):
    # Direct distance check — avoids ASE NeighborList's pair-cutoff doubling
    # (NeighborList uses cutoff[i]+cutoff[j], which would double the threshold)
    idx = [j for j, a in enumerate(atoms) if j != i_tag and a.symbol not in NOBLE_GASES]
    if not idx:
        return 0, []
    _, d = get_distances(
        atoms.positions[i_tag:i_tag+1],
        atoms.positions[idx],
        cell=atoms.cell, pbc=atoms.pbc,
    )
    d = np.array(d).ravel()
    neigh = [idx[k] for k in range(len(idx)) if d[k] < cutoff]
    return len(neigh), neigh

def _min_dist_to_slab(atoms, i_tag):
    idx = [j for j, a in enumerate(atoms) if j != i_tag and a.symbol not in NOBLE_GASES]
    if not idx:
        return np.inf
    _, d = get_distances(atoms.positions[i_tag:i_tag+1], atoms.positions[idx],
                         cell=atoms.cell, pbc=atoms.pbc)
    return float(np.min(d))

# CN cutoff: probe is a neighbor of a slab atom if distance < d1 + margin
d1     = _estimate_first_nn(slab)
cn_cut = d1 + margin
print(f'NN distance: {d1:.3f} Å   CN cutoff: {cn_cut:.3f} Å')
print(f'(NeighborList would have used 2×{cn_cut:.3f} = {2*cn_cut:.3f} Å — too large)')

# place probe 3 Å above void centroid
z_start   = void_pos[2] + 3.0
start_pos = np.array([void_pos[0], void_pos[1], z_start])

atoms = slab.copy() + Atoms(tag_symbol, positions=[start_pos])
i_tag = len(atoms) - 1

frames_drop = []
meta_drop   = []
max_cn      = -1
best_step   = 0
no_improve  = 0

for step in range(int(max_drop / dz) + 1):
    cn, neigh = _tag_cn(atoms, i_tag, cn_cut)
    mind = _min_dist_to_slab(atoms, i_tag)
    z    = float(atoms.positions[i_tag, 2])

    frames_drop.append(atoms.copy())
    meta_drop.append({'step': step, 'z': z, 'cn': cn, 'min_dist': mind})

    if mind < min_clearance:
        print(f'  Stopped at step {step}: min_clearance reached ({mind:.3f} Å)')
        break

    if cn > max_cn:
        max_cn, best_step, no_improve = cn, step, 0
    elif cn > 0:          # only count stability once probe has made surface contact
        no_improve += 1
    if no_improve >= stable_steps:
        print(f'  Stopped at step {step}: CN stable for {stable_steps} steps')
        break

    atoms.positions[i_tag, 2] -= dz

print(f'Drop: {len(frames_drop)} frames   max CN = {max_cn} at step {best_step} (z = {meta_drop[best_step]["z"]:.3f} Å)')

write('panel2_drop_steps.traj', frames_drop)
print('Wrote: panel2_drop_steps.traj')

In [ ]:
# Panel 2 — CN vs z figure
z_vals  = [m['z']  for m in meta_drop]
cn_vals = [m['cn'] for m in meta_drop]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(z_vals, cn_vals, 'o-', color='steelblue', ms=4, lw=1.8, label='CN during drop')
ax.axvline(z_vals[best_step], color='tomato', ls='--', lw=1.5,
           label=f'max CN = {max_cn}')
ax.scatter([z_vals[best_step]], [cn_vals[best_step]],
           c='gold', s=140, zorder=5, edgecolors='k', linewidths=0.8)

# annotate start and end
ax.annotate('probe above surface', xy=(z_vals[0], cn_vals[0]),
            xytext=(z_vals[0] - 0.3, cn_vals[0] + 0.4),
            fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate(f'max CN site\nz = {z_vals[best_step]:.2f} Å',
            xy=(z_vals[best_step], cn_vals[best_step]),
            xytext=(z_vals[best_step] + 0.5, cn_vals[best_step] - 0.8),
            fontsize=8, color='tomato', arrowprops=dict(arrowstyle='->', color='tomato'))

ax.set_xlabel('z of probe (Å)', fontsize=12)
ax.set_ylabel('Coordination number (CN)', fontsize=12)
ax.set_title('Panel 2: CN vs z during noble-gas drop into void', fontsize=12)
ax.invert_xaxis()   # left = high z (above surface), right = low z (deep in void)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('panel2_cn_vs_z.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel2_cn_vs_z.png')

## Panel 3 — Unique sites inside the void

**Method:** Collect all unique probe positions from the drop trajectory (PBC-aware, within `uniq_tol`).  
Each unique position is a candidate adsorption site inside the void.

**Trajectory:** one frame per unique site.  
**Figure:** side-view (x vs z), probe positions coloured by CN.

In [ ]:
# Panel 3 — unique sites trajectory
uniq_frames = []
uniq_meta   = []

for f, m in zip(frames_drop, meta_drop):
    pos_f = f.positions[i_tag]
    keep  = True
    for u in uniq_frames:
        pos_u = u.positions[i_tag]
        _, d  = get_distances([pos_f], [pos_u], cell=f.cell, pbc=f.pbc)
        if float(d) < uniq_tol:
            keep = False
            break
    if keep:
        uniq_frames.append(f.copy())
        uniq_meta.append(m)

print(f'Unique probe positions: {len(uniq_frames)}')
for m in uniq_meta:
    print(f"  step {m['step']:3d}  z = {m['z']:.3f} Å  CN = {m['cn']}")

write('panel3_void_sites.traj', uniq_frames)
print('\nWrote: panel3_void_sites.traj')

In [ ]:
# Panel 3 — side-view matplotlib figure
slab_x = slab.positions[:, 0]
slab_z = slab.positions[:, 2]

cn_all = np.array([m['cn'] for m in uniq_meta], dtype=float)
z_all  = np.array([uniq_frames[k].positions[i_tag, 2] for k in range(len(uniq_frames))])
x_all  = np.array([uniq_frames[k].positions[i_tag, 0] for k in range(len(uniq_frames))])

cmap = plt.cm.plasma
norm = Normalize(vmin=cn_all.min() - 0.5, vmax=cn_all.max() + 0.5)

fig, ax = plt.subplots(figsize=(7, 4))

ax.scatter(slab_x, slab_z, c='lightsteelblue', s=130, edgecolors='gray',
           linewidths=0.5, zorder=2, label='slab atoms')

sc = ax.scatter(x_all, z_all, c=cn_all, cmap=cmap, norm=norm,
                s=200, edgecolors='k', linewidths=0.8,
                marker='D', zorder=4, label='probe positions')
plt.colorbar(sc, ax=ax, label='Coordination number (CN)')

# highlight the max-CN site
best_idx = int(np.argmax(cn_all))
ax.scatter(x_all[best_idx], z_all[best_idx],
           c='gold', s=350, marker='*', edgecolors='k', linewidths=1.0,
           zorder=5, label=f'max CN = {int(cn_all[best_idx])}')

# label each site with its CN
for k in range(len(uniq_frames)):
    ax.annotate(f"CN={int(cn_all[k])}",
                xy=(x_all[k], z_all[k]),
                xytext=(x_all[k] + 0.15, z_all[k] + 0.15),
                fontsize=7, color='k')

ax.set_xlabel('x (Å)', fontsize=12)
ax.set_ylabel('z (Å)', fontsize=12)
ax.set_title('Panel 3: Side view — unique probe sites inside void', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig('panel3_void_sites.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: panel3_void_sites.png')

## Summary

## High-Level Presentation Slide

Standalone schematic — no real data needed, runs independently of cells above.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
import numpy as np

# ── colour palette ────────────────────────────────────────────────────────
C_SLAB  = '#4472C4'
C_RING  = '#C0504D'
C_GOLD  = '#F7B731'
C_PROBE = '#E67E22'
C_GRAY  = '#7F7F7F'
C_LGRAY = '#D0D0D0'
C_BG    = '#F7F9FC'
C_TITLE = '#1A2A4A'

# ── figure + GridSpec ────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 7.5), facecolor='white')
gs  = GridSpec(2, 3, figure=fig,
               left=0.04, right=0.96, top=0.87, bottom=0.07,
               wspace=0.32, hspace=0.30,
               width_ratios=[1.0, 1.0, 1.0],
               height_ratios=[1.25, 1.0])

ax1  = fig.add_subplot(gs[:, 0])   # Step 1 — full height
ax2a = fig.add_subplot(gs[0, 1])   # Step 2 — side-view schematic
ax2b = fig.add_subplot(gs[1, 1])   # Step 2 — CN vs z
ax3  = fig.add_subplot(gs[:, 2])   # Step 3 — full height

for ax in [ax1, ax2a, ax2b, ax3]:
    ax.set_facecolor(C_BG)
    for sp in ax.spines.values():
        sp.set_edgecolor(C_LGRAY)

# ── global title ──────────────────────────────────────────────────────────
fig.text(0.5, 0.97,
         'Defect Surface Site Analysis: Detecting Adsorption Sites Inside Voids',
         ha='center', va='top', fontsize=15, fontweight='bold', color=C_TITLE)
fig.text(0.5, 0.92,
         'Noble-gas probe drop method  |  Pt(111) surface with vacancy  |  PBC-aware distance',
         ha='center', va='top', fontsize=9.5, color=C_GRAY, style='italic')

# ── arrows between panels ─────────────────────────────────────────────────
for xf in [0.365, 0.685]:
    fig.text(xf, 0.50, '→', ha='center', va='center',
             fontsize=30, color=C_GRAY, fontweight='bold')

# ══════════════════════════════════════════════════════════════════════════
# PANEL 1 — top-view hexagonal lattice with vacancy
# ══════════════════════════════════════════════════════════════════════════
a, h = 1.0, np.sqrt(3) / 2

pts = np.array([[col + (row % 2) * 0.5, row * h]
                for row in range(-2, 3) for col in range(-2, 3)])

vac_i  = np.argmin(np.linalg.norm(pts, axis=1))
vac_xy = pts[vac_i]

dv   = np.linalg.norm(pts - vac_xy, axis=1)
ring = np.where((dv > 0.05) & (dv < a * 1.05))[0]

r = 0.36
for i, (x, y) in enumerate(pts):
    if np.linalg.norm([x, y] - vac_xy) > 2.7:
        continue
    if i == vac_i:
        ax1.add_patch(Circle((x, y), r, color=C_LGRAY, ec=C_GRAY,
                             lw=1.0, ls='--', zorder=2))
        ax1.text(x, y, 'V', ha='center', va='center',
                 fontsize=7.5, color=C_GRAY, fontweight='bold')
    elif i in ring:
        ax1.add_patch(Circle((x, y), r, color=C_RING, ec='#8B1A1A',
                             lw=1.2, zorder=3))
        ax1.text(x, y - 0.55, 'deg=5', ha='center',
                 fontsize=5.5, color='#8B1A1A')
    else:
        ax1.add_patch(Circle((x, y), r, color=C_SLAB, ec='#1E4E8C',
                             lw=0.7, zorder=2))

ax1.plot(*vac_xy, marker='*', ms=16, color=C_GOLD, mec='k', mew=0.9, zorder=5)

ax1.set(xlim=(-2.3, 2.3), ylim=(-2.45, 2.6), aspect='equal')
ax1.axis('off')
ax1.set_title('Step 1: Detect Void', fontsize=12, fontweight='bold',
              color=C_TITLE, pad=7)
ax1.text(0, 2.45,
         'Build in-plane neighbor graph\n'
         'Atoms with degree < median\n'
         '→ ring around the vacancy',
         ha='center', va='top', fontsize=9, color=C_TITLE,
         bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=C_LGRAY, alpha=0.9))
ax1.legend(handles=[
    mpatches.Patch(color=C_SLAB,  label='Surface atom'),
    mpatches.Patch(color=C_RING,  label='Undercoordinated (ring)'),
    mpatches.Patch(color=C_LGRAY, label='Vacancy'),
], fontsize=7.5, loc='lower center', framealpha=0.9, edgecolor=C_LGRAY)

# ══════════════════════════════════════════════════════════════════════════
# PANEL 2a — side-view schematic of probe dropping
# ══════════════════════════════════════════════════════════════════════════
layer_z    = [2.5, 1.2, 0.0]
layer_xoff = [0.0, 0.5, 0.0]
r2 = 0.30

for li, (lz, xoff) in enumerate(zip(layer_z, layer_xoff)):
    for col in range(-3, 4):
        x = col + xoff
        if abs(x) > 3.1:
            continue
        is_vac = (li == 0 and col == 0)
        if is_vac:
            ax2a.add_patch(Circle((x, lz), r2, color=C_LGRAY, ec=C_GRAY,
                                  lw=0.8, ls='--', zorder=2))
            ax2a.text(x, lz, 'V', ha='center', va='center',
                      fontsize=6, color=C_GRAY, fontweight='bold')
        else:
            ax2a.add_patch(Circle((x, lz), r2, color=C_SLAB, ec='#1E4E8C',
                                  lw=0.6, zorder=2))

for pz, alpha in [(4.3, 0.2), (3.6, 0.4), (2.9, 0.65)]:
    ax2a.add_patch(Circle((0, pz), 0.22, color=C_PROBE, ec='darkorange',
                          lw=0.8, alpha=alpha, zorder=3))

ax2a.add_patch(Circle((0, 1.2), 0.22, color=C_PROBE, ec='darkorange',
                       lw=1.5, zorder=4))
ax2a.plot(0, 1.2, '*', ms=12, color=C_GOLD, mec='k', mew=0.7, zorder=5)

ax2a.annotate('', xy=(0, 2.95), xytext=(0, 4.45),
              arrowprops=dict(arrowstyle='->', color=C_PROBE, lw=2.2))
ax2a.text(0.38, 3.7, 'dz = 0.1 Å\nper step', fontsize=7.5, color=C_PROBE)
ax2a.text(-2.9, 1.2, '★ max CN', va='center', fontsize=7.5,
          color='#8B0000', fontweight='bold')

ax2a.set(xlim=(-3.1, 3.1), ylim=(-0.55, 4.9), aspect='equal')
ax2a.axis('off')
ax2a.set_title('Step 2: Drop Noble-Gas Probe', fontsize=12,
               fontweight='bold', color=C_TITLE, pad=7)
ax2a.text(0, 4.75, 'Ne probe (orange) descends into vacancy — side view',
          ha='center', fontsize=7.5, color=C_GRAY)

# ══════════════════════════════════════════════════════════════════════════
# PANEL 2b — CN vs z curve
# ══════════════════════════════════════════════════════════════════════════
def _cn_model(z):
    if z > 3.4:   return 0.0
    if z > 2.6:   return 3.0 * (3.4 - z) / (3.4 - 2.6)
    if z > 2.1:   return 3.0
    if z > 1.35:  return 3.0 + 3.0 * (2.1 - z) / (2.1 - 1.35)
    if z > 1.15:  return 6.0
    return max(0, 6.0 - 5.5 * (1.15 - z))

z_ax  = np.linspace(4.5, 0.4, 300)
np.random.seed(7)
cn_ax = np.clip([_cn_model(z) + 0.04 * np.random.randn() for z in z_ax], 0, None)

ax2b.plot(z_ax, cn_ax, color=C_SLAB, lw=2.2)
ax2b.axvline(1.2, color=C_RING, ls='--', lw=1.5)
ax2b.scatter([1.2], [6.0], s=100, c=C_GOLD, ec='k', lw=0.9, zorder=5)
ax2b.invert_xaxis()
ax2b.set_xlabel('z of probe (Å)', fontsize=9)
ax2b.set_ylabel('Coordination number', fontsize=9)
ax2b.set_title('CN vs z during drop', fontsize=10, color=C_TITLE)
ax2b.tick_params(labelsize=8)
ax2b.set_yticks([0, 2, 4, 6])

ax2b.annotate('CN = 0\n(above surface)', xy=(4.1, 0.15),
              fontsize=7, color=C_GRAY, ha='center')
ax2b.annotate('ring\ncontact', xy=(2.7, 3.2),
              fontsize=7, color=C_GRAY, ha='center')
ax2b.annotate('max CN\n(optimal site) ★', xy=(1.2, 6.2),
              fontsize=7.5, color='#8B0000', ha='right')

# ══════════════════════════════════════════════════════════════════════════
# PANEL 3 — unique sites inside the void, coloured by CN
# ══════════════════════════════════════════════════════════════════════════
for li, (lz, xoff) in enumerate(zip(layer_z, layer_xoff)):
    for col in range(-3, 4):
        x = col + xoff
        if abs(x) > 3.1:
            continue
        is_vac = (li == 0 and col == 0)
        if is_vac:
            ax3.add_patch(Circle((x, lz), r2, color=C_LGRAY, ec=C_GRAY,
                                 lw=0.8, ls='--', zorder=2))
        else:
            ax3.add_patch(Circle((x, lz), r2, color=C_SLAB, ec='#1E4E8C',
                                 lw=0.6, zorder=2))

sites = [
    (0.0, 3.8, 0, False),
    (0.0, 3.1, 1, False),
    (0.0, 2.4, 3, False),
    (0.0, 1.75, 4, False),
    (0.0, 1.2,  6, True),
    (0.0, 0.65, 2, False),
]
cmap_p = plt.cm.plasma
norm_p = Normalize(vmin=0, vmax=6)

for x, z, cn, is_best in sites:
    col = cmap_p(norm_p(cn))
    lw  = 1.8 if is_best else 0.8
    ax3.add_patch(Circle((x, z), 0.22, color=col, ec='k', lw=lw, zorder=4))
    ax3.text(0.42, z, f'CN = {cn}', va='center', fontsize=8.5, color=C_TITLE)
    if is_best:
        ax3.plot(x, z, '*', ms=14, color=C_GOLD, mec='k', mew=0.8, zorder=5)
        ax3.text(-0.42, z, '← optimal site', va='center', ha='right',
                 fontsize=8.5, color='#8B0000', fontweight='bold')

# colorbar (right of panel 3)
sm  = plt.cm.ScalarMappable(cmap=cmap_p, norm=norm_p)
sm.set_array([])
cax = fig.add_axes([0.972, 0.24, 0.012, 0.38])
cb  = fig.colorbar(sm, cax=cax)
cb.set_label('CN', fontsize=9)
cb.ax.tick_params(labelsize=8)

ax3.set(xlim=(-3.1, 3.1), ylim=(-0.55, 4.9), aspect='equal')
ax3.axis('off')
ax3.set_title('Step 3: Identify Unique Sites', fontsize=12,
              fontweight='bold', color=C_TITLE, pad=7)
ax3.text(0, 4.75,
         'Unique probe positions coloured by CN',
         ha='center', fontsize=7.5, color=C_GRAY)
ax3.text(0, 4.42,
         'Max CN = optimal adsorption site\n'
         '→ saved to panel3_void_sites.traj',
         ha='center', va='top', fontsize=9, color=C_TITLE,
         bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=C_LGRAY, alpha=0.9))

# ── footer ────────────────────────────────────────────────────────────────
fig.text(0.5, 0.02,
         'Key parameters:  dz = 0.1 Å  |  CN cutoff = d_NN + 0.25 Å  |  '
         'stable_steps = 50 (5 Å patience)  |  min_clearance = 1.3 Å',
         ha='center', fontsize=8.5, color=C_GRAY,
         bbox=dict(boxstyle='round,pad=0.3', fc='#F0F0F0', ec=C_LGRAY, alpha=0.85))

plt.savefig('slide_defect_site_analysis.png', dpi=200,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: slide_defect_site_analysis.png')

In [ ]:
print('=' * 56)
print('Trajectory files for slide panels')
print('=' * 56)
print()
print('  panel1_void_ring.traj    (3 frames)')
print('    Frame 0 : clean slab')
print('    Frame 1 : slab + He at undercoordinated ring atoms')
print('    Frame 2 : Frame 1 + Ne at void centroid')
print()
print(f'  panel2_drop_steps.traj   ({len(frames_drop)} frames)')
print(f'    One frame per dz={dz} Å step, Ne descending into void')
print(f'    Max CN = {max_cn} at step {best_step}')
print()
print(f'  panel3_void_sites.traj   ({len(uniq_frames)} frames)')
print('    One frame per unique probe position (deduped by 0.25 Å)')
print()
print('  PNG figures (for slide):')
print('    panel1_void_ring.png')
print('    panel2_cn_vs_z.png')
print('    panel3_void_sites.png')
print('=' * 56)